In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

import re
import spacy
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

import tensorflow as tf
from tensorflow.keras.layers import LSTM, Dense, Input, TextVectorization, Embedding, concatenate

import joblib

In [2]:
train_ds = pd.read_csv('final-project-danit-ds-3-4/train.csv')

In [3]:
train_ds.head()

,Id,Age,Review_Title,Review,Pos_Feedback_Cnt,Division,Department,Product_Category,Rating,Recommended
0,17274,34,Cute fall/holiday top,Love this top! the quality is magnificent and ...,1,General,Tops,Blouses,5,1
1,5921,35,NaN,NaN,0,General,Tops,Blouses,5,1
2,16479,40,Disappointed,"Sleeves were tight, was difficult to put on ?....",15,General,Tops,Blouses,2,0
3,1925,28,Gorgeous detailing,I never write reviews but this clothe is so fa...,3,General Petite,Clothes,Clothes,5,1
4,5691,39,Cute and comfortable tee!,Love this tshirt! casual but can be clotheed u...,0,General,Tops,Knits,5,1


In [4]:
train_ds.shape

(14091, 10)

In [5]:
nlp = spacy.load('en_core_web_sm')

In [6]:
def text_cleaning(text):

  if not isinstance(text, str):
    return ''

  text_lower = text.lower().strip()

  text_clean = re.sub(r'[^a-z\s]', '', text_lower)

  doc = nlp(text_clean)

  processed_tokens = [token.lemma_ for token in doc]

  return " ".join(processed_tokens)

In [7]:
train_ds_copy = train_ds.copy()

In [8]:
train_ds_copy['Review_Title_cl'] = train_ds_copy['Review_Title'].apply(text_cleaning)

In [9]:
train_ds_copy['Review_cl'] = train_ds_copy['Review'].apply(text_cleaning)

In [10]:
train_ds_copy = train_ds_copy.drop(['Review_Title', 'Review'], axis=1)

In [11]:
scaler = StandardScaler()

In [12]:
train_ds_copy[['Age', 'Pos_Feedback_Cnt']] = scaler.fit_transform(train_ds_copy[['Age', 'Pos_Feedback_Cnt']])

In [13]:
train_ds_copy.head()

,Id,Age,Pos_Feedback_Cnt,Division,Department,Product_Category,Rating,Recommended,Review_Title_cl,Review_cl
0,17274,-0.746854,-0.265497,General,Tops,Blouses,5,1,cute fallholiday top,love this top the quality be magnificent and t...
1,5921,-0.664725,-0.446665,General,Tops,Blouses,5,1,,
2,16479,-0.254081,2.270854,General,Tops,Blouses,2,0,disappoint,sleeve be tight be difficult to put on for t...
3,1925,-1.239628,0.096839,General Petite,Clothes,Clothes,5,1,gorgeous detailing,I never write review but this clothe be so fan...
4,5691,-0.336210,-0.446665,General,Tops,Knits,5,1,cute and comfortable tee,love this tshirt casual but can be clothe up w...


In [14]:
train_ds_copy = train_ds_copy.drop(['Rating', 'Recommended'], axis=1)

In [15]:
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

In [16]:
columns_to_encode = ['Division', 'Department', 'Product_Category']
encoded_data = encoder.fit_transform(train_ds_copy[columns_to_encode])

In [17]:
encoded_df = pd.DataFrame(
    encoded_data, 
    columns=encoder.get_feature_names_out(columns_to_encode),
    index=train_ds_copy.index
)

In [18]:
train_enc_copy = pd.concat([train_ds_copy.drop(columns_to_encode, axis=1), encoded_df], axis=1)

In [19]:
train_enc_copy.head(3)

,Id,Age,Pos_Feedback_Cnt,Review_Title_cl,Review_cl,Division_General,Division_General Petite,Division_Initmates,Division_nan,Department_Bottoms,...,Product_Category_Lounge,Product_Category_Outerwear,Product_Category_Shorts,Product_Category_Skirts,Product_Category_Sleep,Product_Category_Sweaters,Product_Category_Swim,Product_Category_Trend,Product_Category_Trousers,Product_Category_nan
0,17274,-0.746854,-0.265497,cute fallholiday top,love this top the quality be magnificent and t...,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,5921,-0.664725,-0.446665,,,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,16479,-0.254081,2.270854,disappoint,sleeve be tight be difficult to put on for t...,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [20]:
train_enc_copy.shape

(14091, 37)

In [21]:
ds_review_title = train_enc_copy[['Id', 'Review_Title_cl']]
ds_review = train_enc_copy[['Id', 'Review_cl']]
ds_review_title.shape, ds_review.shape

((14091, 2), (14091, 2))

In [22]:
ds_other = train_enc_copy.copy()

In [23]:
ds_other = ds_other.drop(['Review_Title_cl', 'Review_cl'], axis=1)

In [24]:
ds_other.head()

,Id,Age,Pos_Feedback_Cnt,Division_General,Division_General Petite,Division_Initmates,Division_nan,Department_Bottoms,Department_Clothes,Department_Coats,...,Product_Category_Lounge,Product_Category_Outerwear,Product_Category_Shorts,Product_Category_Skirts,Product_Category_Sleep,Product_Category_Sweaters,Product_Category_Swim,Product_Category_Trend,Product_Category_Trousers,Product_Category_nan
0,17274,-0.746854,-0.265497,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,5921,-0.664725,-0.446665,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,16479,-0.254081,2.270854,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1925,-1.239628,0.096839,0.0,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5691,-0.336210,-0.446665,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [25]:
title_calc = ds_review_title.dropna(subset='Review_Title_cl')
review_calc = ds_review.dropna(subset='Review_cl')

In [26]:
text_length_rt = [len(str(text).split()) for text in title_calc['Review_Title_cl']]
mean_length_rt = np.mean(text_length_rt)
max_length_rt = np.max(text_length_rt)
min_length_rt = np.min(text_length_rt)
percent_rt = np.percentile(text_length_rt, 95).astype('int')

print(f'Mean: {mean_length_rt:.2f}'
      f'\nMax: {max_length_rt}'
      f'\nMin: {min_length_rt}'
      f'\n95%: {percent_rt}'
)

Mean: 2.76
Max: 13
Min: 0
95%: 7


In [27]:
text_length_r = [len(str(text).split()) for text in review_calc['Review_cl']]
mean_length_r = np.mean(text_length_r)
max_length_r = np.max(text_length_r)
min_length_r = np.min(text_length_r)
percent_r = np.percentile(text_length_r, 95).astype('int')

print(f'Mean: {mean_length_r:.2f}'
      f'\nMax: {max_length_r}'
      f'\nMin: {min_length_r}'
      f'\n95%: {percent_r}'
)

Mean: 57.41
Max: 115
Min: 0
95%: 101


In [28]:
max_tokens_rt = 10000
sequence_length_rt = percent_rt.item()

In [29]:
max_tokens_r = 20000
sequence_length_r = percent_r.item()

In [30]:
text_vectorization_rt = TextVectorization(
    standardize=None,
    max_tokens=max_tokens_rt,
    output_mode='int',
    output_sequence_length=sequence_length_rt
)

In [31]:
text_vectorization_rt.adapt(ds_review_title['Review_Title_cl'])

In [32]:
text_vectorization_r = TextVectorization(
    standardize=None,
    max_tokens=max_tokens_r,
    output_mode='int',
    output_sequence_length=sequence_length_r
)

In [33]:
text_vectorization_r.adapt(ds_review['Review_cl'])

In [34]:
X_title = text_vectorization_rt(ds_review_title['Review_Title_cl'])
X_title.shape

TensorShape([14091, 7])

In [35]:
X_review = text_vectorization_r(ds_review['Review_cl'])
X_review.shape

TensorShape([14091, 101])

In [36]:
ds_other.shape

(14091, 35)

In [37]:
input_review_title = Input(shape=(None,), name="review_title")
input_review = Input(shape=(None,), name="review")
input_other_columns = Input(shape=(ds_other.shape[1],), name="other_columns")

embedding_review_title = Embedding(input_dim=max_tokens_rt, output_dim=128)(input_review_title)
embedding_review = Embedding(input_dim=max_tokens_r, output_dim=256)(input_review)

lstm_review_title = LSTM(16,name = 'lstm_review_title')(embedding_review_title)
lstm_review = LSTM(32,name = 'lstm_review')(embedding_review)
dense_other_columns = Dense(2, name='dense_other_columns')(input_other_columns)

concat_layer = concatenate([lstm_review_title, lstm_review, dense_other_columns])

rating_pred = Dense(5, activation ='softmax', name="rating")(concat_layer)
recommended_pred = Dense(1, activation ='sigmoid', name="recommended")(concat_layer)

model_lstm = tf.keras.Model(
    inputs=[input_review_title, input_review, input_other_columns],
    outputs=[rating_pred, recommended_pred],
)

model_lstm.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ review_title (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ review (InputLayer)           │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ embedding (Embedding)         │ (None, None, 128)         │       1,280,000 │ review_title[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ embedding_1 (Embedding)       │ (None, None, 256)         │       5,120,000 │ review[0][0]               │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ other_columns (InputLayer)    │ (None, 35)                │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ lstm_review_title (LSTM)      │ (None, 16)                │           9,280 │ embedding[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ lstm_review (LSTM)            │ (None, 32)                │          36,992 │ embedding_1[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_other_columns (Dense)   │ (None, 2)                 │              72 │ other_columns[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ concatenate (Concatenate)     │ (None, 50)                │               0 │ lstm_review_title[0][0],   │
│                               │                           │                 │ lstm_review[0][0],         │
│                               │                           │                 │ dense_other_columns[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ rating (Dense)                │ (None, 5)                 │             255 │ concatenate[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ recommended (Dense)           │ (None, 1)                 │              51 │ concatenate[0][0]          │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 6,446,650 (24.59 MB)

 Trainable params: 6,446,650 (24.59 MB)

 Non-trainable params: 0 (0.00 B)

In [38]:
model_lstm.compile(
    optimizer='adam',
    loss={
        'rating': 'sparse_categorical_crossentropy',
        'recommended': 'binary_crossentropy'
    },
    metrics={
        'rating': 'accuracy',
        'recommended': 'accuracy'
    }
)

In [39]:
y_rat = train_ds['Rating']
y_rec = train_ds['Recommended']

In [40]:
y_rat_upd = y_rat - 1

In [41]:
history_lstm = model_lstm.fit(
    x={
        'review_title': X_title, 
        'review': X_review, 
        'other_columns': ds_other
    },
    y={
        'rating': y_rat_upd, 
        'recommended': y_rec
    },
    epochs=10,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/10
177/177 ━━━━━━━━━━━━━━━━━━━━ 11s 46ms/step - loss: 516.8062 - rating_accuracy: 0.1718 - rating_loss: 491.3401 - recommended_accuracy: 0.6810 - recommended_loss: 23.3751 - val_loss: 38.2843 - val_rating_accuracy: 0.5523 - val_rating_loss: 36.7079 - val_recommended_accuracy: 0.5733 - val_recommended_loss: 0.9032
Epoch 2/10
177/177 ━━━━━━━━━━━━━━━━━━━━ 8s 44ms/step - loss: 14.6897 - rating_accuracy: 0.4677 - rating_loss: 14.0076 - recommended_accuracy: 0.7468 - recommended_loss: 0.6320 - val_loss: 3.7284 - val_rating_accuracy: 0.4484 - val_rating_loss: 3.1981 - val_recommended_accuracy: 0.8138 - val_recommended_loss: 0.5066
Epoch 3/10
177/177 ━━━━━━━━━━━━━━━━━━━━ 8s 45ms/step - loss: 2.7447 - rating_accuracy: 0.5221 - rating_loss: 2.2482 - recommended_accuracy: 0.8153 - recommended_loss: 0.4911 - val_loss: 2.1698 - val_rating_accuracy: 0.5335 - val_rating_loss: 1.5623 - val_recommended_accuracy: 0.6446 - val_recommended_loss: 0.6081
Epoch 4/10
177/177 ━━━━━━━━━━━━━━━━━━━━ 8s 45

In [42]:
loss, rating_loss, recommendation_loss, rating_mae, recommendation_accuracy = model_lstm.evaluate(
    x={
        'review_title': X_title, 
        'review': X_review, 
        'other_columns': ds_other
    },
    y={
        'rating': y_rat_upd, 
        'recommended': y_rec
    },)

print(f'Loss: {loss:.3f}'
      f'\nRating Loss: {rating_loss:.3f}'
      f'\nRating MAE: {rating_mae:.3f}'
      f'\nRecomendation Accuracy: {recommendation_accuracy:.3f}'
      f'\nRecomendation Loss: {recommendation_loss:.3f}'
      )

441/441 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - loss: 0.8526 - rating_accuracy: 0.7430 - rating_loss: 0.6764 - recommended_accuracy: 0.9257 - recommended_loss: 0.1766
Loss: 0.853
Rating Loss: 0.676
Rating MAE: 0.743
Recomendation Accuracy: 0.926
Recomendation Loss: 0.177


In [312]:
# joblib.dump(model_lstm, 'combo_lstm.joblib')

['combo_lstm.joblib']

# Test

In [43]:
test_ds = pd.read_csv('final-project-danit-ds-3-4/test.csv')

In [44]:
test_ds.head()

,Id,Age,Review_Title,Review,Pos_Feedback_Cnt,Division,Department,Product_Category
0,21403,53,Magnificent clothe!,"In contrast to the other reviewer, i love this...",4,General,Clothes,Clothes
1,22553,51,Shapeless tent,I tried this on in the store and it was huge. ...,2,General,Clothes,Clothes
2,17436,59,Versatile and then some,"I thought this was a fun piece to have, but di...",1,General,Bottoms,Trousers
3,4293,48,So simple but so cute!,I bought the multi-color stripe and it is ador...,1,General,Clothes,Clothes
4,20149,46,Magnificent simple tank,The wide strap style is very flattering. this ...,0,Initmates,Intimate,Layering


In [45]:
test_ds_copy = test_ds.copy()

In [46]:
test_ds_copy['Review_Title'] = test_ds_copy['Review_Title'].apply(text_cleaning)

In [47]:
test_ds_copy['Review'] = test_ds_copy['Review'].apply(text_cleaning)

In [48]:
test_ds_copy[['Age', 'Pos_Feedback_Cnt']] = scaler.transform(test_ds_copy[['Age', 'Pos_Feedback_Cnt']])

In [49]:
test_encoded_data = encoder.transform(test_ds_copy[columns_to_encode])

In [50]:
test_encoded_df = pd.DataFrame(
    test_encoded_data, 
    columns=encoder.get_feature_names_out(columns_to_encode),
    index=test_ds_copy.index
)

In [51]:
test_enc_copy = pd.concat([test_ds_copy.drop(columns_to_encode, axis=1), test_encoded_df], axis=1)

In [52]:
test_enc_copy.head(3)

,Id,Age,Review_Title,Review,Pos_Feedback_Cnt,Division_General,Division_General Petite,Division_Initmates,Division_nan,Department_Bottoms,...,Product_Category_Lounge,Product_Category_Outerwear,Product_Category_Shorts,Product_Category_Skirts,Product_Category_Sleep,Product_Category_Sweaters,Product_Category_Swim,Product_Category_Trend,Product_Category_Trousers,Product_Category_nan
0,21403,0.813596,magnificent clothe,in contrast to the other reviewer I love this ...,0.278007,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,22553,0.649338,shapeless tent,I try this on in the store and it be huge I co...,-0.084329,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,17436,1.306370,versatile and then some,I think this be a fun piece to have but do not...,-0.265497,1.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [53]:
test_review_title = test_enc_copy[['Id', 'Review_Title']]
test_review = test_enc_copy[['Id', 'Review']]
test_review_title.shape, test_review.shape

((9395, 2), (9395, 2))

In [54]:
test_other = test_enc_copy.drop(['Review', 'Review_Title'], axis=1)

In [55]:
test_other.shape

(9395, 35)

In [56]:
X_title_test = text_vectorization_rt(test_review_title['Review_Title'])
X_title_test.shape

TensorShape([9395, 7])

In [57]:
X_review_test = text_vectorization_r(test_review['Review'])
X_review_test.shape

TensorShape([9395, 101])

In [58]:
y_rat_prob, y_rec_prob = model_lstm.predict(x={
        'review_title': X_title_test, 
        'review': X_review_test, 
        'other_columns': test_other
    })

294/294 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step


In [59]:
final_rating = np.argmax(y_rat_prob, axis=1) + 1

In [60]:
final_rec = (y_rec_prob > 0.5).astype(int)

In [61]:
sub_ds = pd.read_csv('final-project-danit-ds-3-4/sample_submission.csv')

In [62]:
sub_ds.head()

,Id,Rating,Recommended
0,21403,2,1
1,22553,5,0
2,17436,1,1
3,4293,1,0
4,20149,4,0


In [63]:
sub_ds['Rating'] = final_rating
sub_ds['Recommended'] = final_rec

In [64]:
sub_ds.head()

,Id,Rating,Recommended
0,21403,5,1
1,22553,3,0
2,17436,5,1
3,4293,5,1
4,20149,5,1


In [ ]:
sub_ds.to_csv('submission.csv', index=False)